# 05 · Grad-CAM RSNA Knee (Fase 5, entregable 5)

Settings: **Accelerator None** (CPU basta; no entrena) e Internet Off.
Add Input: el Dataset `rsna-knee-subset-v1` y el output del notebook `03_train_rsna_knee` (pestaña Your Work → Notebooks).

Método: Grad-CAM propio sobre la última etapa convolucional del codificador de cada plano. Hay un mapa por ventana
2.5D; se muestra la ventana con más energía del mapa. Los mapas de un plano se normalizan con el máximo del estudio.

Casos: para cada hallazgo, los 3 verdaderos positivos dorados con mayor probabilidad y el falso positivo con mayor
probabilidad (dorado = 0). Hallazgos: ACL, Medial Meniscus, Medial OA, Effusion y Baker's.

Salida en `/kaggle/working/gradcam/`: una figura por modelo y hallazgo, una figura resumen y `cam_cases.csv`.
Las figuras muestran imágenes de pacientes: van solo al informe y nunca a GitHub ni a un Dataset público.

In [ ]:
import json, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

SKIP = {"train_series", "test_series", "vol"}  # carpetas con miles de archivos: no se recorren

def find(rel, root="/kaggle/input", max_depth=7):
    # Busca la ruta relativa `rel` (p. ej. 'resnet18_base/best.pt') sin entrar en SKIP.
    root = Path(root)
    for dirpath, dirnames, _ in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if d not in SKIP)
        if (Path(dirpath) / rel).exists():
            return Path(dirpath) / rel
    raise FileNotFoundError(f"no encuentro {rel} en {root} (¿falta Add Input?)")

print("inputs montados:", sorted(str(p.relative_to("/kaggle/input")) for p in Path("/kaggle/input").glob("*/*/*")))

DATA = find("subset_studies.csv").parent
RUNS = {m: find(f"{m}/best.pt").parent for m in ["resnet18_base", "scratch_base"]}
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("/kaggle/working/gradcam"); OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(os.cpu_count())
print(DATA); print(RUNS); print("dispositivo", DEV, "· hilos", torch.get_num_threads())

## 1. Código del repo (copiado tal cual por `notebooks/build_05_gradcam.py`)

In [ ]:
CODE = Path('/tmp/rsnak')
FILES = {
"src/__init__.py": "",
"src/data/__init__.py": "",
"src/data/select_subset.py": "\"\"\"Fase 3: política de etiquetas, selección de series, muestreo y split del subconjunto.\n\nSolo usa pandas/numpy, así que se prueba en el Mac sin imágenes. El notebook\n`02_subset_rsna_knee` incluye este archivo tal cual.\n\"\"\"\nimport numpy as np\nimport pandas as pd\n\nLABELS = [\n    \"ACL\", \"MCL\", \"Medial Meniscus\", \"Lateral Meniscus\", \"Medial OA\", \"Lateral OA\",\n    \"PF OA\", \"Effusion\", \"Synovitis\", \"Baker's\", \"Contusion\", \"Fracture\",\n]\n\n# Política fijada en la Fase 2 antes de abrir test: solo POS es 1.\n# OJO: en la columna `<label>` de weak_labels.csv MILD figura como 1; por eso se usa `__state`.\nSTATE_TO_Y = {\"POS\": 1, \"MILD\": 0, \"MIN\": 0, \"NEG\": 0, \"NONE\": 0, \"UNC\": -1, \"WEAK\": -1}\n\n# Roles de serie: (plano, combinación preferida, respaldo) con combinación = (fluid, FS).\nROLES = {\"sag\": (\"Sagittal\", (1, 1), (0, 0)), \"cor\": (\"Coronal\", (1, 1), (0, 0))}\nTHIN_MM = 1.5      # por debajo se considera adquisición 3D\nTARGET_SLICES = 30\n\n\ndef apply_policy(weak):\n    \"\"\"weak_labels.csv → una fila por estudio con y_, mask_, conf_, score_ y state_ por etiqueta.\n\n    y_ vale 1/0 (los -1 quedan como 0) y mask_ vale 0 donde el estado es UNC o WEAK.\n    \"\"\"\n    out = {\"StudyInstanceUID\": weak[\"StudyInstanceUID\"].values}\n    for lab in LABELS:\n        state = weak[f\"{lab}__state\"]\n        unknown = set(state.unique()) - set(STATE_TO_Y)\n        if unknown:\n            raise ValueError(f\"{lab}: estados desconocidos {unknown}\")\n        y = state.map(STATE_TO_Y)\n        out[f\"y_{lab}\"] = y.clip(lower=0).astype(int).values\n        out[f\"mask_{lab}\"] = (y != -1).astype(int).values\n        out[f\"conf_{lab}\"] = weak[f\"{lab}__conf\"].values\n        out[f\"score_{lab}\"] = weak[f\"{lab}__score\"].values\n        out[f\"state_{lab}\"] = state.values\n    return pd.DataFrame(out)\n\n\ndef _pick(cands):\n    \"\"\"Entre varias series candidatas: 2D primero, luego cortes cerca de 30, luego mayor matriz.\"\"\"\n    c = cands.assign(\n        _is2d=(cands[\"SliceThickness\"].fillna(0) >= 2).astype(int),\n        _dist=(cands[\"n_slices\"] - TARGET_SLICES).abs(),\n    )\n    c = c.sort_values([\"_is2d\", \"_dist\", \"Rows\"], ascending=[False, True, False])\n    return c.iloc[0]\n\n\ndef choose_series(catalog):\n    \"\"\"series_catalog.csv → una fila por estudio con la serie elegida para cada rol.\n\n    Columnas por rol: <rol>_series, <rol>_fallback (1 si se usó fluid=0 FS=0),\n    <rol>_n_slices, <rol>_thick, <rol>_is3d.\n    \"\"\"\n    rows = []\n    for uid, g in catalog.groupby(\"StudyInstanceUID\", sort=False):\n        r = {\"StudyInstanceUID\": uid}\n        for role, (plane, pref, back) in ROLES.items():\n            pick, fallback = None, 0\n            for i, (fl, fs) in enumerate((pref, back)):\n                m = g[(g[\"Anatomical_Plane\"] == plane) & (g[\"Fluid_Sensitive\"] == fl)\n                      & (g[\"Fat_Suppression\"] == fs) & (g[\"n_slices\"] > 0)]\n                if len(m):\n                    pick, fallback = _pick(m), i\n                    break\n            if pick is None:\n                r.update({f\"{role}_series\": None, f\"{role}_fallback\": np.nan,\n                          f\"{role}_n_slices\": np.nan, f\"{role}_thick\": np.nan, f\"{role}_is3d\": np.nan})\n            else:\n                th = pick[\"SliceThickness\"]\n                r.update({f\"{role}_series\": pick[\"SeriesInstanceUID\"], f\"{role}_fallback\": fallback,\n                          f\"{role}_n_slices\": int(pick[\"n_slices\"]), f\"{role}_thick\": th,\n                          f\"{role}_is3d\": int(pd.notna(th) and th < THIN_MM)})\n        g0 = g.iloc[0]\n        r[\"Manufacturer\"] = g0.get(\"Manufacturer\")\n        r[\"FieldStrength\"] = g0.get(\"FieldStrength\")\n        rows.append(r)\n    return pd.DataFrame(rows)\n\n\ndef eligible_for_training(series):\n    \"\"\"Estudios no dorados utilizables: sagital y coronal presentes y ninguna 3D.\"\"\"\n    s = series\n    ok = s[\"sag_series\"].notna() & s[\"cor_series\"].notna() & (s[\"sag_is3d\"] == 0) & (s[\"cor_is3d\"] == 0)\n    return set(s.loc[ok, \"StudyInstanceUID\"])\n\n\ndef sample_studies(labels, eligible, n=500, min_pos=60, seed=42):\n    \"\"\"Muestreo estratificado: primero asegura `min_pos` positivos (POS) por etiqueta,\n    empezando por la más rara, y completa hasta `n` al azar.\n\n    Devuelve la lista de UIDs elegidos (orden reproducible).\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    pool = labels[labels[\"StudyInstanceUID\"].isin(eligible)].set_index(\"StudyInstanceUID\")\n    pool = pool.loc[rng.permutation(pool.index.values)]\n    chosen = []\n    chosen_set = set()\n    order = sorted(LABELS, key=lambda lab: pool[f\"y_{lab}\"].sum())\n    for lab in order:\n        have = int(pool.loc[chosen, f\"y_{lab}\"].sum()) if chosen else 0\n        if have >= min_pos:\n            continue\n        cands = [u for u in pool.index[pool[f\"y_{lab}\"] == 1] if u not in chosen_set]\n        for u in cands[: min_pos - have]:\n            chosen.append(u)\n            chosen_set.add(u)\n    if len(chosen) > n:\n        raise ValueError(f\"min_pos={min_pos} exige {len(chosen)} estudios > n={n}\")\n    rest = [u for u in pool.index if u not in chosen_set]\n    chosen += rest[: n - len(chosen)]\n    return chosen\n\n\ndef split_train_val(labels, uids, val_frac=0.2, seed=42):\n    \"\"\"Split por estudio. Reparte los positivos de cada etiqueta (de la más rara a la más\n    común) para que val tenga cerca de `val_frac` de ellos. Devuelve {uid: 'train'|'val'}.\"\"\"\n    rng = np.random.default_rng(seed)\n    df = labels.set_index(\"StudyInstanceUID\").loc[list(uids)]\n    df = df.loc[rng.permutation(df.index.values)]\n    n_val = int(round(val_frac * len(df)))\n    split = {}\n    order = sorted(LABELS, key=lambda lab: df[f\"y_{lab}\"].sum())\n    for lab in order:\n        pos = df.index[df[f\"y_{lab}\"] == 1]\n        target = int(round(val_frac * len(pos)))\n        have = sum(split.get(u) == \"val\" for u in pos)\n        for u in pos:\n            if u in split:\n                continue\n            if have < target and sum(v == \"val\" for v in split.values()) < n_val:\n                split[u] = \"val\"\n                have += 1\n            else:\n                split[u] = \"train\"\n    n_now = sum(v == \"val\" for v in split.values())\n    for u in df.index:\n        if u not in split:\n            split[u] = \"val\" if n_now < n_val else \"train\"\n            n_now += split[u] == \"val\"\n    return split\n\n\ndef prevalence(labels, uids, by=None):\n    \"\"\"Positivos y % por etiqueta (y_ == 1) sobre los estudios dados; opcionalmente por grupo.\"\"\"\n    df = labels[labels[\"StudyInstanceUID\"].isin(set(uids))]\n    def one(d):\n        return pd.Series({**{lab: int(d[f\"y_{lab}\"].sum()) for lab in LABELS}, \"n\": len(d)})\n    if by is None:\n        t = one(df).to_frame(\"pos\")\n        t[\"pct\"] = (t[\"pos\"] / len(df) * 100).round(1)\n        t.loc[\"n\", \"pct\"] = np.nan\n        return t\n    return df.groupby(by).apply(one)\n",
"src/models/__init__.py": "",
"src/models/cnn_scratch.py": "\"\"\"Modelo 1: CNN 2D propia que codifica un corte 2.5D (3 cortes contiguos como canales).\n\nBloques conv-BN-ReLU con atajo residual, cuatro etapas con stride 2 y GAP.\nImplementada desde cero (sin torchvision) y entrenada sin pesos previos.\n\"\"\"\nimport torch\nfrom torch import nn\n\n\nclass ConvBNReLU(nn.Sequential):\n    def __init__(self, c_in, c_out, k=3, stride=1):\n        super().__init__(\n            nn.Conv2d(c_in, c_out, k, stride, k // 2, bias=False),\n            nn.BatchNorm2d(c_out),\n            nn.ReLU(inplace=True),\n        )\n\n\nclass ResBlock(nn.Module):\n    \"\"\"Dos convoluciones 3×3 con atajo; el atajo usa conv 1×1 cuando cambia la forma.\"\"\"\n\n    def __init__(self, c_in, c_out, stride=1):\n        super().__init__()\n        self.conv1 = ConvBNReLU(c_in, c_out, 3, stride)\n        self.conv2 = nn.Sequential(nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm2d(c_out))\n        self.short = nn.Identity()\n        if stride != 1 or c_in != c_out:\n            self.short = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))\n        self.act = nn.ReLU(inplace=True)\n\n    def forward(self, x):\n        return self.act(self.conv2(self.conv1(x)) + self.short(x))\n\n\nclass ScratchCNN(nn.Module):\n    \"\"\"(N, 3, H, W) → (N, out_dim). Con 256×256 la última etapa queda en 8×8.\"\"\"\n\n    def __init__(self, in_ch=3, widths=(32, 64, 128, 256), blocks_per_stage=2):\n        super().__init__()\n        self.stem = nn.Sequential(ConvBNReLU(in_ch, widths[0], 5, 2), ConvBNReLU(widths[0], widths[0]))\n        stages, c = [], widths[0]\n        for w in widths:\n            blocks = [ResBlock(c, w, stride=2)] + [ResBlock(w, w) for _ in range(blocks_per_stage - 1)]\n            stages.append(nn.Sequential(*blocks))\n            c = w\n        self.stages = nn.Sequential(*stages)\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.out_dim = c\n        self.cam_layer = self.stages[-1]  # capa objetivo para Grad-CAM\n        for m in self.modules():\n            if isinstance(m, nn.Conv2d):\n                nn.init.kaiming_normal_(m.weight, mode=\"fan_out\", nonlinearity=\"relu\")\n            elif isinstance(m, nn.BatchNorm2d):\n                nn.init.ones_(m.weight)\n                nn.init.zeros_(m.bias)\n\n    def forward(self, x):\n        return torch.flatten(self.pool(self.stages(self.stem(x))), 1)\n",
"src/models/backbone_tl.py": "\"\"\"Modelo 2: backbone de torchvision preentrenado en ImageNet, sin su capa de clasificación.\n\nSolo se toman los pesos del extractor. La agregación por cortes y por plano y la cabeza\nde 12 salidas están en `mil.py` y son propias.\n\"\"\"\nimport torch\nfrom torch import nn\nfrom torchvision import models as tvm\n\nBACKBONES = {\n    \"resnet18\": (tvm.resnet18, 512),\n    \"resnet34\": (tvm.resnet34, 512),\n    \"resnet50\": (tvm.resnet50, 2048),\n    \"efficientnet_b0\": (tvm.efficientnet_b0, 1280),\n}\n\n\nclass TVBackbone(nn.Module):\n    \"\"\"(N, 3, H, W) → (N, out_dim).\"\"\"\n\n    def __init__(self, name=\"resnet18\", pretrained=True):\n        super().__init__()\n        if name not in BACKBONES:\n            raise ValueError(f\"backbone desconocido {name}; opciones {list(BACKBONES)}\")\n        fn, self.out_dim = BACKBONES[name]\n        net = fn(weights=\"DEFAULT\" if pretrained else None)\n        if name.startswith(\"resnet\"):\n            self.features = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool,\n                                          net.layer1, net.layer2, net.layer3, net.layer4)\n        else:\n            self.features = net.features\n        self.cam_layer = self.features[-1]\n        self.pool = nn.AdaptiveAvgPool2d(1)\n\n    def forward(self, x):\n        return torch.flatten(self.pool(self.features(x)), 1)\n",
"src/models/mil.py": "\"\"\"Modelo a nivel de estudio: codificador 2D compartido por cortes, agregación por plano y cabeza.\n\nEntrada por plano: (B, D, 3, H, W), es decir D ventanas 2.5D de 3 cortes contiguos.\nCada ventana pasa por el mismo codificador, los D vectores se agregan (max, media o\natención, como en MRNet y en el MIL con atención de Ilse et al.), los planos se\nconcatenan y una capa lineal da 12 logits (una sigmoide independiente por hallazgo).\n\"\"\"\nimport torch\nfrom torch import nn\n\nfrom src.models.cnn_scratch import ScratchCNN\n\n\nclass SlicePool(nn.Module):\n    def __init__(self, dim, mode=\"attn\", hidden=128):\n        super().__init__()\n        self.mode = mode\n        if mode == \"attn\":\n            self.att = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\n        elif mode not in (\"max\", \"mean\"):\n            raise ValueError(f\"pool desconocido {mode}\")\n\n    def forward(self, h):\n        \"\"\"h: (B, D, dim) → (B, dim), pesos por corte (B, D) o None.\"\"\"\n        if self.mode == \"max\":\n            return h.max(1).values, None\n        if self.mode == \"mean\":\n            return h.mean(1), None\n        a = torch.softmax(self.att(h).squeeze(-1).float(), dim=1).to(h.dtype)\n        return (a.unsqueeze(-1) * h).sum(1), a\n\n\ndef make_encoder(cfg):\n    if cfg[\"model\"] == \"scratch\":\n        return ScratchCNN(widths=tuple(cfg.get(\"widths\", (32, 64, 128, 256))))\n    from src.models.backbone_tl import TVBackbone  # torchvision solo hace falta aquí\n    return TVBackbone(cfg[\"model\"], pretrained=cfg.get(\"pretrained\", True))\n\n\nclass StudyNet(nn.Module):\n    def __init__(self, cfg, n_out=12):\n        super().__init__()\n        self.planes = list(cfg[\"planes\"])\n        self.encoders = nn.ModuleDict({p: make_encoder(cfg) for p in self.planes})\n        dim = next(iter(self.encoders.values())).out_dim\n        self.pools = nn.ModuleDict({p: SlicePool(dim, cfg.get(\"pool\", \"attn\")) for p in self.planes})\n        self.head = nn.Sequential(nn.Dropout(cfg.get(\"dropout\", 0.3)), nn.Linear(dim * len(self.planes), n_out))\n\n    def forward(self, x, return_attn=False):\n        feats, attn = [], {}\n        for p in self.planes:\n            v = x[p]\n            b, d = v.shape[:2]\n            h = self.encoders[p](v.flatten(0, 1)).view(b, d, -1)\n            z, a = self.pools[p](h)\n            feats.append(z)\n            attn[p] = a\n        logits = self.head(torch.cat(feats, 1))\n        return (logits, attn) if return_attn else logits\n\n    def backbone_parameters(self):\n        return [q for e in self.encoders.values() for q in e.parameters()]\n\n    def set_backbone_trainable(self, flag):\n        \"\"\"Congela o libera los codificadores. Congelados, su BN queda en modo eval.\"\"\"\n        for e in self.encoders.values():\n            for q in e.parameters():\n                q.requires_grad_(flag)\n            e.train(flag and self.training)\n        self._bb_frozen = not flag\n\n    def train(self, mode=True):\n        super().train(mode)\n        if getattr(self, \"_bb_frozen\", False):\n            for e in self.encoders.values():\n                e.eval()\n        return self\n",
"src/train/__init__.py": "",
"src/train/data.py": "\"\"\"Fase 4: lectura de rsna-knee-subset-v1 y armado de ventanas 2.5D por estudio.\n\nCada volumen (n, 256, 256) uint8 se reduce a D ventanas de `window` cortes contiguos\n(los cortes van como canales). Los centros se reparten de forma uniforme en la serie;\nen entrenamiento se desplazan todos juntos al azar como aumento.\n\nObjetivos por split:\n- train y val: y_ (política POS = 1) con pesos según la estrategia de ruido.\n- test: los 58 dorados (gold_*), todos con peso 1.\n\"\"\"\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom src.data.select_subset import LABELS\n\n\ndef load_studies(data_dir):\n    df = pd.read_csv(Path(data_dir) / \"subset_studies.csv\")\n    assert df.StudyInstanceUID.is_unique\n    return df\n\n\ndef label_weights(df, noise=\"mask\", highconf_thr=0.9):\n    \"\"\"Peso por estudio y etiqueta (N, 12) según la estrategia de manejo de ruido.\n\n    - mask: 1 donde la etiqueta es conocida (UNC y WEAK quedan en 0).\n    - conf: la confianza del extractor (POS 0.9 a 0.95, NEG 0.9, MIN 0.7, NONE y MILD 0.6).\n    - highconf: 1 solo si conf >= highconf_thr; con 0.9 quedan POS y NEG explícitos y se\n      descartan los \"no mencionados\".\n    \"\"\"\n    mask = df[[f\"mask_{l}\" for l in LABELS]].to_numpy(np.float32)\n    conf = df[[f\"conf_{l}\" for l in LABELS]].to_numpy(np.float32)\n    if noise == \"mask\":\n        return mask\n    if noise == \"conf\":\n        return mask * conf\n    if noise == \"highconf\":\n        return mask * (conf >= highconf_thr - 1e-6)\n    raise ValueError(f\"noise desconocido {noise}\")\n\n\ndef targets(df, split, noise=\"mask\", highconf_thr=0.9):\n    \"\"\"(y, w) como arreglos (N, 12) float32 para las filas de `df` (todas del mismo split).\"\"\"\n    if split == \"test\":\n        y = df[[f\"gold_{l}\" for l in LABELS]].to_numpy(np.float32)\n        assert not np.isnan(y).any(), \"test sin etiqueta dorada\"\n        return y, np.ones_like(y)\n    y = df[[f\"y_{l}\" for l in LABELS]].to_numpy(np.float32)\n    return y, label_weights(df, noise, highconf_thr)\n\n\ndef window_indices(n, n_windows=16, window=3, jitter=0.0, rng=None):\n    \"\"\"Índices (n_windows, window) de cortes. jitter en fracción del paso entre centros.\"\"\"\n    half = window // 2\n    lo, hi = half, max(half, n - 1 - half)\n    centers = np.linspace(lo, hi, n_windows)\n    if jitter and rng is not None:\n        step = (hi - lo) / max(n_windows - 1, 1)\n        centers = centers + rng.uniform(-jitter, jitter) * step\n    centers = np.clip(np.round(centers), lo, hi).astype(int)\n    idx = centers[:, None] + np.arange(-half, half + 1)[None, :]\n    return np.clip(idx, 0, n - 1)\n\n\nclass KneeStudies(Dataset):\n    \"\"\"Devuelve {\"x\": {plano: uint8 (D, window, H, W)}, \"y\": (12,), \"w\": (12,), \"i\": índice}.\"\"\"\n\n    PLANE_FILE = {\"sag\": \"sag\", \"cor\": \"cor\"}\n\n    def __init__(self, df, data_dir, split, planes=(\"sag\",), n_windows=16, window=3,\n                 noise=\"mask\", highconf_thr=0.9, train=False, cache=None, seed=0):\n        self.df = df.reset_index(drop=True)\n        self.dir = Path(data_dir)\n        self.planes, self.n_windows, self.window, self.train = list(planes), n_windows, window, train\n        self.y, self.w = targets(self.df, split, noise, highconf_thr)\n        self.cache = {} if cache is None else cache\n        self.rng = np.random.default_rng(seed)\n        for uid in self.df.StudyInstanceUID:\n            for p in self.planes:\n                key = f\"{uid}_{self.PLANE_FILE[p]}\"\n                if key not in self.cache:\n                    self.cache[key] = np.load(self.dir / \"vol\" / f\"{key}.npy\")\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, i):\n        uid = self.df.StudyInstanceUID.iat[i]\n        x = {}\n        for p in self.planes:\n            v = self.cache[f\"{uid}_{self.PLANE_FILE[p]}\"]\n            idx = window_indices(len(v), self.n_windows, self.window,\n                                 jitter=0.5 if self.train else 0.0, rng=self.rng)\n            x[p] = torch.from_numpy(np.ascontiguousarray(v[idx]))\n        return {\"x\": x, \"y\": torch.from_numpy(self.y[i]), \"w\": torch.from_numpy(self.w[i]), \"i\": i}\n\n\ndef worker_init(worker_id):\n    info = torch.utils.data.get_worker_info()\n    info.dataset.rng = np.random.default_rng(torch.initial_seed() % 2**32)\n",
"src/train/augment.py": "\"\"\"Aumentos en GPU, iguales para todos los cortes de un mismo estudio y plano.\n\nSin flips: en sagital el flip horizontal cambia anterior por posterior y en coronal\ncambia medial por lateral, lo que alteraría Medial y Lateral Meniscus/OA.\nSe usan rotación leve, escala, traslación, contraste, brillo y gamma.\n\"\"\"\nimport math\n\nimport torch\nimport torch.nn.functional as F\n\nMEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)\nSTD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)\n\n\ndef to_float(x):\n    \"\"\"uint8 (B, D, 3, H, W) → float en [0, 1].\"\"\"\n    return x.float().div_(255)\n\n\ndef normalize(x):\n    \"\"\"Estandarización con la media y desviación de ImageNet (para ambos modelos).\"\"\"\n    return (x - MEAN.to(x.device)) / STD.to(x.device)\n\n\ndef random_affine(x, max_deg=10, scale=(0.9, 1.1), max_shift=0.06):\n    b, d, c, h, w = x.shape\n    dev = x.device\n    ang = (torch.rand(b, device=dev) * 2 - 1) * math.radians(max_deg)\n    s = torch.empty(b, device=dev).uniform_(*scale)\n    t = (torch.rand(b, 2, device=dev) * 2 - 1) * max_shift * 2\n    cos, sin = torch.cos(ang) / s, torch.sin(ang) / s\n    theta = torch.stack([torch.stack([cos, -sin, t[:, 0]], 1), torch.stack([sin, cos, t[:, 1]], 1)], 1)\n    theta = theta.repeat_interleave(d, 0)\n    flat = x.reshape(b * d, c, h, w)\n    grid = F.affine_grid(theta, flat.shape, align_corners=False)\n    return F.grid_sample(flat, grid, mode=\"bilinear\", padding_mode=\"zeros\", align_corners=False).view(b, d, c, h, w)\n\n\ndef random_intensity(x, contrast=(0.8, 1.2), brightness=0.1, gamma=(0.8, 1.25)):\n    b = x.shape[0]\n    shape = (b, 1, 1, 1, 1)\n    dev = x.device\n    g = torch.exp(torch.empty(shape, device=dev).uniform_(math.log(gamma[0]), math.log(gamma[1])))\n    x = x.clamp(0, 1).pow(g)\n    k = torch.empty(shape, device=dev).uniform_(*contrast)\n    m = x.mean(dim=(1, 2, 3, 4), keepdim=True)\n    x = (x - m) * k + m + torch.empty(shape, device=dev).uniform_(-brightness, brightness)\n    return x.clamp(0, 1)\n\n\ndef prepare(x, train):\n    \"\"\"uint8 en el dispositivo → tensor normalizado, con aumentos si train.\"\"\"\n    x = to_float(x)\n    if train:\n        x = random_intensity(random_affine(x))\n    return normalize(x)\n",
"src/interpret/__init__.py": "",
"src/interpret/gradcam.py": "\"\"\"Fase 5 (entregable 5): Grad-CAM para StudyNet, por plano y por ventana 2.5D.\n\nPara la etiqueta k se toma el gradiente del logit k respecto de la salida de `cam_layer`\n(última etapa del codificador de cada plano). El peso de cada canal es la media espacial\ndel gradiente y el mapa es ReLU(suma de canales ponderados), uno por ventana. Los mapas de\nun plano se normalizan con el máximo del estudio, así que las ventanas que casi no aportan\n(poca atención) quedan oscuras. Implementación propia (Selvaraju et al., 2017).\n\"\"\"\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nfrom src.models.mil import StudyNet\nfrom src.train.augment import prepare\nfrom src.train.data import window_indices\n\n\ndef load_model(ckpt_path, device=\"cpu\"):\n    ck = torch.load(ckpt_path, map_location=device, weights_only=False)\n    cfg = dict(ck[\"cfg\"], pretrained=False)  # los pesos vienen del checkpoint\n    model = StudyNet(cfg).to(device).eval()\n    model.load_state_dict(ck[\"state_dict\"])\n    return model, cfg\n\n\ndef study_input(vols, cfg):\n    \"\"\"{plano: volumen uint8 (n, H, W)} → ({plano: uint8 (1, D, 3, H, W)}, {plano: índices (D, 3)}).\"\"\"\n    x, idx = {}, {}\n    for p in cfg[\"planes\"]:\n        v = vols[p]\n        idx[p] = window_indices(len(v), cfg[\"n_windows\"], cfg[\"window\"])\n        x[p] = torch.from_numpy(np.ascontiguousarray(v[idx[p]]))[None]\n    return x, idx\n\n\nclass GradCAM:\n    def __init__(self, model):\n        self.model = model\n        self.acts, self.grads, self.handles = {}, {}, []\n        for p, enc in model.encoders.items():\n            self.handles.append(enc.cam_layer.register_forward_hook(self._hook(p)))\n\n    def _hook(self, plane):\n        def fn(module, inp, out):\n            self.acts[plane] = out\n            out.register_hook(lambda g: self.grads.__setitem__(plane, g))\n        return fn\n\n    def remove(self):\n        for h in self.handles:\n            h.remove()\n\n    def __call__(self, x, label_idx):\n        \"\"\"x: {plano: uint8 (1, D, 3, H, W)} en el dispositivo del modelo.\n\n        Devuelve prob, {plano: cams (D, H, W) en [0, 1]}, {plano: energía por ventana (D,)},\n        {plano: pesos de atención (D,) o None}.\n        \"\"\"\n        self.model.zero_grad(set_to_none=True)\n        with torch.enable_grad():\n            xin = {p: prepare(v.to(next(self.model.parameters()).device), train=False) for p, v in x.items()}\n            logits, attn = self.model(xin, return_attn=True)\n            logits[0, label_idx].backward()\n        cams, energy = {}, {}\n        for p in x:\n            a, g = self.acts[p].detach().float(), self.grads[p].float()\n            w = g.mean(dim=(2, 3), keepdim=True)\n            cam = F.relu((w * a).sum(1, keepdim=True))\n            cam = F.interpolate(cam, size=x[p].shape[-2:], mode=\"bilinear\", align_corners=False)[:, 0]\n            energy[p] = cam.sum(dim=(1, 2)).cpu().numpy()\n            cams[p] = (cam / cam.max().clamp_min(1e-8)).cpu().numpy()\n        prob = torch.sigmoid(logits[0, label_idx]).item()\n        att = {p: (a[0].detach().float().cpu().numpy() if a is not None else None) for p, a in attn.items()}\n        return prob, cams, energy, att\n"
}
for rel, src in FILES.items():
    p = CODE / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(src)
sys.path.insert(0, str(CODE))
from src.data.select_subset import LABELS
from src.interpret.gradcam import GradCAM, load_model, study_input
print(len(FILES), "archivos en", CODE)

## 2. Selección de casos (desde `preds_test.csv` del notebook 03)

In [ ]:
TARGETS = ["ACL", "Medial Meniscus", "Medial OA", "Effusion", "Baker's"]
N_TP = 3
studies = pd.read_csv(DATA / "subset_studies.csv").set_index("StudyInstanceUID")

def pick_cases(preds, lab):
    tp = preds[preds[f"y_{lab}"] == 1].nlargest(N_TP, f"p_{lab}").assign(kind="VP")
    fp = preds[preds[f"y_{lab}"] == 0].nlargest(1, f"p_{lab}").assign(kind="FP")
    return pd.concat([tp, fp])

preds = {m: pd.read_csv(d / "preds_test.csv") for m, d in RUNS.items()}
# Rango percentil medio de los positivos dorados entre los 58 (0.5 = azar, 1 = todos arriba)
for m, p in preds.items():
    print(m, {lab: round(float(p[f"p_{lab}"].rank(pct=True)[p[f"y_{lab}"] == 1].mean()), 2) for lab in TARGETS})

## 3. Grad-CAM

In [ ]:
def overlay(ax, img, cam=None, title=""):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    if cam is not None:
        ax.imshow(cam, cmap="jet", alpha=0.35, vmin=0, vmax=1)
    ax.set_title(title, fontsize=8); ax.axis("off")

rows = []
summary = {}
for m, run in RUNS.items():
    model, cfg = load_model(run / "best.pt", DEV)
    gc = GradCAM(model)
    gold = preds[m].set_index("StudyInstanceUID")
    for lab in TARGETS:
        j = LABELS.index(lab)
        cases = pick_cases(preds[m], lab)
        planes = cfg["planes"]
        fig, axes = plt.subplots(len(cases), 2 * len(planes) + 1, figsize=(3 * (2 * len(planes) + 1), 3 * len(cases)))
        for r, case in enumerate(cases.itertuples()):
            uid = case.StudyInstanceUID
            vols = {p: np.load(DATA / "vol" / f"{uid}_{p}.npy") for p in planes}
            x, idx = study_input(vols, cfg)
            t0 = time.time()
            prob, cams, energy, att = gc(x, j)
            print(f"{m} · {lab} · {case.kind} · …{uid[-6:]} · p={prob:.2f} · {time.time()-t0:.1f} s", flush=True)
            ax = axes[r]
            for c, p in enumerate(planes):
                k = int(np.argmax(energy[p]))
                s = int(idx[p][k][len(idx[p][k]) // 2])
                share = float(energy[p][k] / max(energy[p].sum(), 1e-8))
                overlay(ax[2 * c], vols[p][s], None, f"{p} corte {s}/{len(vols[p])}")
                overlay(ax[2 * c + 1], vols[p][s], cams[p][k], f"Grad-CAM {lab} (ventana {k})")
                rows.append({"model": m, "label": lab, "StudyInstanceUID": uid, "kind": case.kind,
                             "prob": prob, "gold": int(gold.at[uid, f"y_{lab}"]),
                             "lang": studies.at[uid, "lang"], "plane": p, "window": k, "slice": s, "n_slices": len(vols[p]),
                             "energy_share": share,
                             "attn_argmax": None if att[p] is None else int(np.argmax(att[p])),
                             "attn_max": None if att[p] is None else float(np.max(att[p]))})
                if m == "resnet18_base" and case.kind == "VP" and lab not in summary:
                    summary[lab] = (uid, prob)
            a = ax[-1]
            for p in planes:
                e = energy[p] / max(energy[p].sum(), 1e-8)
                a.plot(e, marker="o", label=f"CAM {p}")
                if att[p] is not None:
                    a.plot(att[p], ls="--", label=f"atención {p}")
            a.set_title(f"{case.kind} · p={prob:.2f} · {studies.at[uid, 'lang']} · …{uid[-6:]}", fontsize=8)
            a.set_xlabel("ventana", fontsize=8); a.legend(fontsize=6)
        fig.suptitle(f"{m} · {lab}", fontsize=11)
        plt.tight_layout()
        fname = OUT / f"{m}_{lab.replace(' ', '_').replace(chr(39), '')}.png"
        fig.savefig(fname, dpi=110); plt.show(); plt.close(fig)
    gc.remove()

cases_df = pd.DataFrame(rows)
cases_df.to_csv(OUT / "cam_cases.csv", index=False)
print(cases_df.groupby(["model", "label", "kind"]).prob.mean().round(3).unstack())

## 4. Figura resumen para el informe (ResNet-18, mejor verdadero positivo por hallazgo, ambos planos)

In [ ]:
model, cfg = load_model(RUNS["resnet18_base"] / "best.pt", DEV)
gc = GradCAM(model)
fig, axes = plt.subplots(len(TARGETS), 2 * len(cfg["planes"]), figsize=(12, 3 * len(TARGETS)))
for r, lab in enumerate(TARGETS):
    uid, _ = summary[lab]
    vols = {p: np.load(DATA / "vol" / f"{uid}_{p}.npy") for p in cfg["planes"]}
    x, idx = study_input(vols, cfg)
    prob, cams, energy, att = gc(x, LABELS.index(lab))
    for c, p in enumerate(cfg["planes"]):
        k = int(np.argmax(energy[p])); s = int(idx[p][k][1])
        overlay(axes[r, 2 * c], vols[p][s], None, f"{lab} · {p} · p={prob:.2f}")
        overlay(axes[r, 2 * c + 1], vols[p][s], cams[p][k], "Grad-CAM")
gc.remove()
plt.tight_layout(); fig.savefig(OUT / "gradcam_summary_resnet18.png", dpi=130); plt.show()

## 5. Guardar

**Save Version → Save & Run All** (CPU, sin cola de GPU). Luego bajar `gradcam/` completo a
`rsna-knee-weak/outputs/gradcam_v1/` para revisar las figuras y escribir la discusión de plausibilidad anatómica.